In [1]:
!pip install pyspark

In [3]:
with open("/content/transactions.csv", "w") as f:
    f.write("""user_id,date,amount
1,2026-05-01,500
1,2026-05-05,300
1,2026-05-15,5000
2,2026-05-03,400
2,2026-05-10,350
2,2026-05-20,4500
3,2026-05-07,200
3,2026-05-18,250
""")

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, month

# Create Spark session
spark = SparkSession.builder \
    .appName("Expense Monitoring System") \
    .getOrCreate()

# Load CSV
df = spark.read.csv("/content/transactions.csv", header=True, inferSchema=True)

print("Original Data:")
df.show()

# Add month column
df = df.withColumn("month", month(col("date")))

# Group by user and calculate total monthly spend
monthly_spend = df.groupBy("user_id", "month") \
    .agg(sum("amount").alias("total_spend"))

print("Monthly Spend Per User:")
monthly_spend.show()

# Calculate average expense
avg_amount = df.select(avg("amount")).collect()[0][0]

print("Average Expense:", avg_amount)

# Detect unusual large expenses
unusual = df.filter(col("amount") > avg_amount * 2)

print("Potential Unusual Spending:")
unusual.show()

# Stop Spark
spark.stop()

Original Data:
+-------+----------+------+
|user_id|      date|amount|
+-------+----------+------+
|      1|2026-05-01|   500|
|      1|2026-05-05|   300|
|      1|2026-05-15|  5000|
|      2|2026-05-03|   400|
|      2|2026-05-10|   350|
|      2|2026-05-20|  4500|
|      3|2026-05-07|   200|
|      3|2026-05-18|   250|
+-------+----------+------+

Monthly Spend Per User:
+-------+-----+-----------+
|user_id|month|total_spend|
+-------+-----+-----------+
|      3|    5|        450|
|      2|    5|       5250|
|      1|    5|       5800|
+-------+-----+-----------+

Average Expense: 1437.5
Potential Unusual Spending:
+-------+----------+------+-----+
|user_id|      date|amount|month|
+-------+----------+------+-----+
|      1|2026-05-15|  5000|    5|
|      2|2026-05-20|  4500|    5|
+-------+----------+------+-----+

